In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer # Import SimpleImputer
from sklearn.preprocessing import StandardScaler # Import StandardScaler
improt matplotlib as mp

df=pd.read_csv("/content/drive/MyDrive/RAP_Dataset_Public_deident.csv")
df.isnull().sum()


,0
Female_1,0
White_1,0
Cohort_1,0
Major_d_2,0
Major_d_3,0
Major_d_4,0
Matching_GPA_1,0
Rank_d_2,0
Rank_d_3,0
WVRes_d_1,0


In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# 1. Data Loading & Binary Encoding
df = pd.read_csv("/content/drive/MyDrive/RAP_Dataset_Public_deident.csv").replace(-99, np.nan)
df_ml = df.dropna(subset=['sMSAT_2']).copy()

# Binary Target: 1 if Excellent (>=6.0), 0 if Not
df_ml['Is_Excellent'] = (df_ml['sMSAT_2'] >= 6.0).astype(int)

# 2. Preprocessing
inputs = ['Female_1', 'White_1', 'Matching_GPA_1', 'SATtotEquiv_1', 'PellGrantRecipient_R',
          'Major_d_2', 'Major_d_3', 'Major_d_4', 'gendermatch', 'ethnicitymatch',
          'PriorAflMent_1', 'RAP', 'FMNTi_1r', 'MentorPsychSocSupport_w1',
          'MentorInstrumentSupport_w1', 'MentorRoleModelingSupport_w1', 'sMSAT_1', 'sMSIM_1']

X = SimpleImputer(strategy='median').fit_transform(df_ml[inputs])
X_scaled = StandardScaler().fit_transform(X)
y = df_ml['Is_Excellent']

# Split data (stratified ensures proportional balance of Excellent/Not Excellent)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# 3. Train Models
# Logistic Regression
lr_clf = LogisticRegression(class_weight='balanced', random_state=42)
lr_clf.fit(X_train, y_train)

# Random Forest (Achieves ~81% Accuracy)
rf_clf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
rf_clf.fit(X_train, y_train)

# Gradient Boosting
gb_clf = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_clf.fit(X_train, y_train)

# 4. Results
models = {"Logistic Regression": lr_clf, "Random Forest": rf_clf, "Gradient Boosting": gb_clf}
for name, model in models.items():
    score = accuracy_score(y_test, model.predict(X_test))
    print(f"{name} Binary Accuracy: {score:.4f}")

Logistic Regression Binary Accuracy: 0.8095
Random Forest Binary Accuracy: 0.7619
Gradient Boosting Binary Accuracy: 0.7619


In [20]:
import numpy as np
from sklearn.impute import SimpleImputer # Import SimpleImputer
from sklearn.preprocessing import StandardScaler # Import StandardScaler

# Re-create and fit the scaler based on the original data processing steps
# This is necessary because the original scaler object was not saved in the previous cell.
# We fit it on the full 'X' data (df_ml[inputs]) to match the original scaling.
# df_ml and inputs are available from the kernel state.
imputer = SimpleImputer(strategy='median')
X_imputed_full = imputer.fit_transform(df_ml[inputs])
scaler = StandardScaler()
scaler.fit(X_imputed_full)

# Define the two examples with the correct number of features (18)
# Order must match the 'inputs' list used in training
high_potential = np.array([[1, 1, 3.8, 1300, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 6.0, 1]])
high_risk = np.array([[0, 0, 2.5, 1000, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1.0, 1]])

# Scale the examples (Very important for Logistic Regression/SVM)
hp_scaled = scaler.transform(high_potential)
hr_scaled = scaler.transform(high_risk)

# Get Predictions (using Random Forest as example)
print(f"High Potential Prediction: {rf_clf.predict(hp_scaled)[0]} (Excellent)")
print(f"High Risk Prediction: {rf_clf.predict(hr_scaled)[0]} (Not Excellent)")

# Get Probabilities (Shows how 'sure' the AI is)
prob_hp = rf_clf.predict_proba(hp_scaled)[0][1]
print(f"Confidence for High Potential being Excellent: {prob_hp*100:.1f}%")

High Potential Prediction: 1 (Excellent)
High Risk Prediction: 0 (Not Excellent)
Confidence for High Potential being Excellent: 57.0%


In [22]:
import numpy as np

# A sample student profile (1 row, 24 features)
test = np.array([[
    1.0,   # Female_1 (Yes)
    1.0,   # White_1 (Yes)
    3.8,   # Matching_GPA_1 (GPA of 3.8)
    1200,  # SATtotEquiv_1 (SAT score)
    0.0,   # PellGrantRecipient_R (No)
    1.0,   # WVRes_d_1 (Yes)
    0.0,   # Legacy_R_1 (No)
    1.0,   # Major_d_2 (CS/Engineering/Math)
    0.0,   # Major_d_3 (Social Sci - No)
    0.0,   # Major_d_4 (Humanities - No)
    0.0,   # Rank_d_2 (Junior - No)
    1.0,   # Rank_d_3 (Senior - Yes)
    1.0,   # gendermatch (Yes, same gender as mentor)
    1.0,   # ethnicitymatch (Yes, same ethnicity as mentor)
    0.0,   # PriorAflMent_1 (No prior relationship)
    1.0,   # RAP (Part of the apprenticeship program)
    5.0,   # FMNTi_1r (5 hours per week with mentor)
    1.0,   # SomeRes_1 (Research intensity level 1)
    0.0,   # MoreRes_1 (Research intensity level 2 - No)
    6.5,   # MentorPsychSocSupport_w1 (High emotional support)
    6.0,   # MentorInstrumentSupport_w1 (High task help)
    6.8,   # MentorRoleModelingSupport_w1 (High role modeling)
    6.5,   # sMSAT_1 (High early satisfaction)
    6.2    # sMSIM_1 (High early psychological similarity)
]])
prediction=model.predict(test)
print(prediction)
print(f"MENTORSHIP OUTCOME PREDICTION")
if prediction[0]==2:
  print("Status: EXCELLENT - High psychological alignment detected.")
elif prediction[0]==1:
   print("Status: STABLE - Standard professional relationship.")
else:
   print("Status: AT RISK - Low similarity/support early indicators.")


ValueError: X has 24 features, but GradientBoostingClassifier is expecting 18 features as input.